# FuzzyMFD — worked examples

**Membership functions, fuzzification and defuzzification in R**, following T. J. Ross (2000), *Fuzzy Systems in Medicine*, Springer, 49–78.

This notebook runs on an **R** runtime. In Colab choose *Runtime → Change runtime type → R*.

Author: Mohamad Yamen AL Mohamad (Department of Statistics, University of Tabriz).

Contents:
1. Installation
2. Fuzzy sets and membership functions
3. Features of a membership function
4. Fuzzification
5. Rank ordering
6. Inductive reasoning by entropy minimisation
7. Defuzzification
8. Tests

## 1. Installation

In [ ]:
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
remotes::install_github("yamenetoo/Statistical-Repository", subdir = "FuzzyMFD")
library(FuzzyMFD)

## 2. Fuzzy sets and membership functions

A fuzzy set is stored as an exact piecewise-linear membership function. Figure 1.1: the crisp set A = [5, 7] and the fuzzy set H of heights near 6 feet.

In [ ]:
hs <- ross2000_sets("height")
membership(hs$A, c(6, 4.99))
membership(hs$H, c(5.917, 6, 7))
plot_fuzzy_sets(hs, xlab = "height (ft)")

In [ ]:
x <- seq(0, 80, by = 10)
cbind(x, tri = mf_triangular(x, 0, 20, 40), trap = mf_trapezoidal(x, 10, 20, 40, 60),
      gauss = round(mf_gaussian(x, 40, 10), 3))

## 3. Features of a membership function (Figures 1.2–1.5)

In [ ]:
A <- fs_trapezoidal(1, 3, 5, 8, name = "A")
summary(A)
plot(A)

In [ ]:
convex <- fuzzy_set(c(0, 2, 5), c(0, 1, 0))
nonconvex <- fuzzy_set(c(0, 2, 3, 4, 6), c(0, 1, 0.7, 0.9, 0))
c(is_convex(convex), is_convex(nonconvex))
AB <- fuzzy_intersection(fs_triangular(0, 2, 5), fs_trapezoidal(2, 4, 6, 8))
is_convex(AB)
plot_fuzzy_sets(list(A = fs_triangular(0, 2, 5), B = fs_trapezoidal(2, 4, 6, 8), "A and B" = AB))

## 4. Fuzzification (Figures 1.6–1.7)

In [ ]:
R <- fuzzify(10, spread = 0.01, relative = TRUE, name = "reading")
fuzzy_support(R)
v <- ross2000_sets("voltage")
agreement(v$low, 5.8)
agreement(v$medium, fuzzify(7.2, 1))

## 5. Rank ordering (Table 1.1, Figure 1.9)

In [ ]:
data(ross_colors)
ro <- rank_ordering(ross_colors)
ro
plot(ro)

## 6. Inductive reasoning by entropy minimisation (Tables 1.2–1.5, Figures 1.13–1.14)

In [ ]:
data(ross_ellipse)
ross_ellipse
entropy_threshold(ross_ellipse$x, ross_ellipse$class, candidates = c(0.7, 1.0, 1.5, 3.45))

In [ ]:
res <- inductive_mf(ross_ellipse$x, ross_ellipse$class, levels = 2, range = c(0, 100))
res$partition
plot_fuzzy_sets(res$sets, xlab = "x = a/b")

## 7. Defuzzification (Section 5.2)

The three fuzzy measurements of mean arterial pressure and their union. The exact centroid is 4.953 mm Hg (the chapter reports 4.9).

In [ ]:
B <- ross2000_sets("map")
c(centroid = defuzzify(B, "centroid"), weighted = defuzzify(B, "weighted"),
  meanmax = defuzzify(B, "meanmax"), max_first = defuzzify(B, "max"))
plot_fuzzy_sets(list(union = fuzzy_union(B)), legend = FALSE, xlab = "MAP (mm Hg)")
abline(v = c(defuzzify(B, "centroid"), defuzzify(B, "weighted"), defuzzify(B, "meanmax")), lty = 2:4)

## 8. Tests

In [ ]:
install.packages("testthat")
dir <- tempfile(); dir.create(dir)
for (f in c("test-fuzzyset.R", "test-assignment.R", "test-defuzzification.R"))
  download.file(paste0("https://raw.githubusercontent.com/yamenetoo/Statistical-Repository/main/FuzzyMFD/tests/testthat/", f), file.path(dir, f))
testthat::test_dir(dir)